# Notebook 03 — EDA & Point-in-Time Feature Integration (MVP)

**Goal:** inspect real interactions, verify property/category coverage at each prediction cutoff, add category hierarchy and historical user-category affinity, and export richer feature tables for Notebook 04.

**Input:** Notebook 02's `data/processed/notebook02.duckdb`, raw CSVs, and report. **Output:** `data/processed/{train,validation,test}_enriched_examples.parquet`, `data/reports/notebook03_feature_report.json`.

Run top-to-bottom in VS Code. This notebook does **not** train a model. Do not interpret sampled weak negatives as actual ad impressions or CTR.

## 1. Setup and verify Notebook 02 outputs
Close any other active DuckDB connection to the same file before running if you get a file-lock error.

In [10]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import duckdb
from IPython.display import display

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p/'data'/'processed'/'notebook02.duckdb').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Cannot find data/processed/notebook02.duckdb. Run Notebook 02 first.')
OUT, REPORT = ROOT/'data'/'processed', ROOT/'data'/'reports'
DB = OUT/'notebook02.duckdb'
con = duckdb.connect(str(DB))
con.execute("SET memory_limit='4GB'")
con.execute('SET threads=4')
q = lambda p: "'" + p.as_posix().replace("'", "''") + "'"
splits = ('train','validation','test')
required = [f'{s}_{suffix}' for s in splits for suffix in ('examples','history','item_features','item_popularity')]
existing = {x[0] for x in con.execute('SHOW TABLES').fetchall()}
assert all(x in existing for x in required), f'Missing Notebook 02 tables: {set(required)-existing}'
report02_path = REPORT/'notebook02_integration_temporal_report.json'
assert report02_path.exists(), 'Notebook 02 JSON report missing'
report02 = json.loads(report02_path.read_text(encoding='utf-8'))
print('ROOT:',ROOT,'DuckDB:',duckdb.__version__)
print('Cutoffs:',report02['cutoffs_utc'])

ROOT: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks DuckDB: 1.5.5
Cutoffs: {'train': '2015-07-31 19:47:53.596000+00:00', 'validation': '2015-08-18 18:21:27.439000+00:00', 'test': '2015-09-04 07:47:49.447000+00:00'}


## 2. Lightweight EDA
Event types, user activity, and item popularity are descriptive. The full event log includes future observations, so **never** use full-period popularity as a training feature.

In [11]:
event_dist = con.execute('SELECT event,COUNT(*) AS events,COUNT(DISTINCT user_id) AS users,COUNT(DISTINCT item_id) AS items FROM events_raw GROUP BY 1 ORDER BY events DESC').df()
display(event_dist)
user_activity = con.execute('SELECT COUNT(*) AS users, MIN(n) AS min_events, MEDIAN(n) AS median_events, MAX(n) AS max_events FROM (SELECT user_id,COUNT(*) n FROM events_raw GROUP BY 1)').df()
item_activity = con.execute('SELECT COUNT(*) AS items, MIN(n) AS min_events, MEDIAN(n) AS median_events, MAX(n) AS max_events FROM (SELECT item_id,COUNT(*) n FROM events_raw GROUP BY 1)').df()
display(user_activity,item_activity)
assert set(event_dist.event) == {'view','addtocart','transaction'}

,event,events,users,items
0,view,2664312,1404179,234838
1,addtocart,69332,37722,23903
2,transaction,22457,11719,12025


,users,min_events,median_events,max_events
0,1407580,1,1.0,7757


,items,min_events,median_events,max_events
0,235061,1,3.0,3412


## 3. Inspect property types and temporal category mappings
`categoryid` is verified in Notebook 02. Count the most common property names without pivoting 20M rows into a wide matrix.

In [12]:
property_top = con.execute('SELECT property,COUNT(*) AS records,COUNT(DISTINCT item_id) AS distinct_items FROM props_all GROUP BY 1 ORDER BY records DESC LIMIT 20').df()
display(property_top)
cat_property_rows = con.execute("SELECT COUNT(*) FROM props_all WHERE property='categoryid'").fetchone()[0]
print('categoryid property rows:',cat_property_rows)
assert cat_property_rows > 0

,property,records,distinct_items
0,888,3000398,417053
1,790,1790516,417053
2,available,1503639,417053
3,categoryid,788214,417053
4,6,631471,409065
5,283,597419,417053
6,776,574220,407305
7,678,481966,417019
8,364,476486,417053
9,202,448938,414217


categoryid property rows: 788214


## 4. Build category ancestry lookup
Verify full hierarchy cycles, then create `root_category_id` and `category_depth` per category. These are static hierarchy metadata; item-to-category assignment is still taken from point-in-time snapshots.

In [13]:
tree = con.execute('SELECT category_id,parent_id FROM category_tree').df()
parent = {int(r.category_id): (None if pd.isna(r.parent_id) else int(r.parent_id)) for r in tree.itertuples(index=False)}
assert len(parent) == len(tree), 'Duplicate category IDs'
assert all(p is None or p in parent for p in parent.values()), 'Orphan parent reference'

def get_ancestry(cat):
    visited=set(); current=cat; depth=0
    while parent[current] is not None:
        if current in visited: raise ValueError(f'Category hierarchy cycle at {current}')
        visited.add(current)
        current=parent[current]
        depth+=1
    return current,depth

rows = [(cat,*get_ancestry(cat)) for cat in parent]
ancestry = pd.DataFrame(rows,columns=['category_id','root_category_id','category_depth'])
con.register('ancestry_input',ancestry)
con.execute('CREATE OR REPLACE TABLE category_ancestry AS SELECT * FROM ancestry_input')
print('Hierarchy checked: no cycles; root categories:',ancestry.root_category_id.nunique())
display(ancestry.category_depth.value_counts().sort_index().rename_axis('depth').reset_index(name='categories'))

Hierarchy checked: no cycles; root categories: 25


,depth,categories
0,0,25
1,1,174
2,2,702
3,3,665
4,4,90
5,5,13


## 5. Snapshot-specific category coverage
Only metadata with `ts <= cutoff` was included by Notebook 02. Calculate coverage among event items and actual sampled examples, separately by split.

In [14]:
coverage=[]
for s in splits:
    coverage.append(con.execute(f"""SELECT '{s}' AS split, COUNT(*) AS examples,
      COUNT(*) FILTER (WHERE category_id IS NOT NULL) AS categorized_examples,
      COUNT(DISTINCT item_id) AS unique_items,
      COUNT(DISTINCT item_id) FILTER (WHERE category_id IS NOT NULL) AS categorized_items
      FROM {s}_examples""").df())
coverage_df=pd.concat(coverage,ignore_index=True)
coverage_df['example_category_fraction']=coverage_df.categorized_examples/coverage_df.examples
coverage_df['unique_item_category_fraction']=coverage_df.categorized_items/coverage_df.unique_items
display(coverage_df)
assert coverage_df.example_category_fraction.between(0,1).all()

,split,examples,categorized_examples,unique_items,categorized_items,example_category_fraction,unique_item_category_fraction
0,train,37448,35236,7618,6892,0.940931,0.904699
1,validation,36080,34095,7406,6904,0.944983,0.932217
2,test,32856,30811,6867,6224,0.937759,0.906364


## 6. Historical user-category affinity
For each cutoff, join 30-day user interactions to that cutoff's **known** item category. Calculate counts of interactions in the candidate's category, using only events before the cutoff. Missing categories remain missing (no future backfill).

In [15]:
for s in splits:
    con.execute(f"""CREATE OR REPLACE TABLE {s}_user_category AS
        SELECT h.user_id,f.category_id,SUM(h.n_events)::BIGINT AS user_category_30d_events,
               COUNT(DISTINCT h.item_id)::BIGINT AS user_category_30d_items
        FROM {s}_history h JOIN {s}_item_features f USING(item_id)
        WHERE f.category_id IS NOT NULL
        GROUP BY 1,2""")
    con.execute(f"""CREATE OR REPLACE TABLE {s}_category_popularity AS
        SELECT f.category_id,SUM(h.n_events)::BIGINT AS category_30d_events
        FROM {s}_history h JOIN {s}_item_features f USING(item_id)
        WHERE f.category_id IS NOT NULL
        GROUP BY 1""")
    print(s, 'user-category pairs:',con.execute(f'SELECT COUNT(*) FROM {s}_user_category').fetchone()[0])

train user-category pairs: 364040
validation user-category pairs: 332893
test user-category pairs: 294763


## 7. Join enriched features and export
Keep the original labels and sampled candidates unchanged. `user_category_affinity` is historical category events divided by all historical user events. Unknown category is explicitly flagged.

In [16]:
for s in splits:
    con.execute(f"""CREATE OR REPLACE TABLE {s}_enriched_examples AS
      SELECT e.*,
        a.root_category_id,a.category_depth,
        COALESCE(uc.user_category_30d_events,0)::BIGINT AS user_category_30d_events,
        COALESCE(uc.user_category_30d_items,0)::BIGINT AS user_category_30d_items,
        COALESCE(cp.category_30d_events,0)::BIGINT AS category_30d_events,
        CASE WHEN e.category_id IS NULL THEN 1 ELSE 0 END AS missing_category_flag,
        COALESCE(uc.user_category_30d_events,0)::DOUBLE / NULLIF(e.user_30d_events,0) AS user_category_affinity
      FROM {s}_examples e
      LEFT JOIN category_ancestry a ON e.category_id=a.category_id
      LEFT JOIN {s}_user_category uc ON e.user_id=uc.user_id AND e.category_id=uc.category_id
      LEFT JOIN {s}_category_popularity cp ON e.category_id=cp.category_id""")
    con.execute(f"COPY {s}_enriched_examples TO {q(OUT/f'{s}_enriched_examples.parquet')} (FORMAT PARQUET)")
    print(s, con.execute(f'SELECT COUNT(*) AS rows, SUM(label) AS positives FROM {s}_enriched_examples').df().to_dict('records'))
display(con.execute('SELECT * FROM train_enriched_examples LIMIT 5').df())

train [{'rows': 37448, 'positives': 9362.0}]
validation [{'rows': 36080, 'positives': 9020.0}]
test [{'rows': 32856, 'positives': 8214.0}]


,user_id,item_id,label,cutoff_ts,user_30d_events,user_30d_unique_items,user_recency_ms,item_30d_events,known_property_count,category_id,parent_category_id,root_category_id,category_depth,user_category_30d_events,user_category_30d_items,category_30d_events,missing_category_flag,user_category_affinity
0,140357,20981,1,1438372073596,6.0,5,224224731,131.0,22,406,872,1600,2,1,1,2128,0,0.166667
1,1297062,192847,1,1438372073596,1652.0,1038,148494508,21.0,25,445,1308,140,3,3,1,371,0,0.001816
2,1331304,100282,0,1438372073596,3.0,3,1264317106,229.0,44,1483,561,395,2,1,1,17004,0,0.333333
3,1264591,385355,0,1438372073596,1.0,1,234677031,78.0,24,1051,955,140,3,1,1,19204,0,1.000000
4,797414,74603,1,1438372073596,2.0,1,711227556,19.0,27,196,1667,140,3,2,1,6826,0,1.000000


## 8. Verify label preservation and historical feature invariants
Check unique pairs, preserved labels, valid affinity, and no category mismatch with the point-in-time item snapshot. These checks do not prove that the offline negative sampling is unbiased.

In [17]:
checks={}
for s in splits:
    orig=con.execute(f'SELECT COUNT(*),SUM(label) FROM {s}_examples').fetchone()
    new=con.execute(f'SELECT COUNT(*),SUM(label),COUNT(DISTINCT (user_id,item_id)) FROM {s}_enriched_examples').fetchone()
    assert orig == new[:2],f'Row/label counts changed in {s}: {orig} vs {new}'
    assert new[0]==new[2],f'Duplicate user-item pair in {s}'
    bad=con.execute(f"""SELECT COUNT(*) FROM {s}_enriched_examples WHERE
      user_category_affinity < 0 OR user_category_affinity > 1.00000001
      OR (category_id IS NULL AND missing_category_flag != 1)
      OR (category_id IS NOT NULL AND root_category_id IS NULL)""").fetchone()[0]
    assert bad==0,f'Invalid category features in {s}: {bad}'
    # Original item feature snapshot already restricts properties to <= cutoff.
    cutoff=int(con.execute(f'SELECT MIN(cutoff_ts) FROM {s}_examples').fetchone()[0])
    future=con.execute(f"""SELECT COUNT(*) FROM props_all WHERE ts>{cutoff} AND ts<={cutoff+7*86400000}""").fetchone()[0]
    checks[s]={'examples':int(new[0]),'positives':int(new[1]),'unique_pairs':int(new[2]),'cutoff_ts':cutoff,'future_property_rows_excluded_by_snapshot_logic':int(future)}
print('Sanity checks passed')
display(pd.DataFrame.from_dict(checks,orient='index'))

Sanity checks passed


,examples,positives,unique_pairs,cutoff_ts,future_property_rows_excluded_by_snapshot_logic
train,37448,9362,37448,1438372073596,529315
validation,36080,9020,36080,1439922087439,571339
test,32856,8214,32856,1441352869447,566183


## 9. Export feature report

In [18]:
def clean_json(x):
    if isinstance(x,dict):return {str(k):clean_json(v) for k,v in x.items()}
    if isinstance(x,list):return [clean_json(v) for v in x]
    if isinstance(x,(np.integer,)):return int(x)
    if isinstance(x,(np.floating,)):return None if not np.isfinite(x) else float(x)
    return x
report03={
    'source':'Notebook 02 point-in-time tables; all four original CSVs contribute through these tables',
    'property_category_rows':int(cat_property_rows),
    'category_hierarchy':{'nodes':len(parent),'roots':int(ancestry.root_category_id.nunique()),'max_depth':int(ancestry.category_depth.max()),'cycle_check':'passed'},
    'example_category_coverage':coverage_df.to_dict('records'),
    'splits':checks,
    'limitations':['Properties used here: categoryid and count of known property names; additional property values reserved for feature ablation.',
                   'Coverage for all event items from Notebook 02 is not point-in-time coverage.',
                   'Sampled weak negatives are not impression-level negatives.',
                   'Notebook 02 uses one cutoff per split and capped positive pairs.',
                   'Offline ranking evaluation needs retrieval-generated candidate sets.']
}
path=REPORT/'notebook03_feature_report.json'
path.write_text(json.dumps(clean_json(report03),indent=2,allow_nan=False),encoding='utf-8')
print('Saved:',path)
print('Next: Notebook 04 — Popularity Retrieval')
con.close()

Saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\data\reports\notebook03_feature_report.json
Next: Notebook 04 — Popularity Retrieval


## Handoff to Notebook 04

Use the historical `train_item_popularity`, `validation_item_popularity`, and `test_item_popularity` tables for **time-aware** popularity retrieval, not popularity computed over the entire event log. Use the enriched examples only for supervised ranking experiments later; sampled candidate AUC is **not** a substitute for full-candidate Recall@K.

**Current limitations:** category hierarchy checks do not guarantee item metadata correctness; unknown category mappings remain null. The property feature contribution is still limited and will be measured with ablation later.